## 요약
- 기존 전처리 코드에서는 정수 인코딩한 token들도 scaling을 진행함
- transformer에서 embedding model을 사용하려면 정수 값으로 입력 받아야 함
- 또한 중고 도서 데이터가 아닌 도서 정보만 사용
- 이에 맞게 코드 수정

In [1]:
import os, natsort, re
from tqdm import tqdm
import time, random

In [2]:
from itertools import repeat, chain

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_squared_log_error as msle

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats("png2x")
# 테마 설정: "default", "classic", "dark_background", "fivethirtyeight", "seaborn"
mpl.style.use("fivethirtyeight")
# 이미지가 레이아웃 안으로 들어오도록 함
mpl.rcParams.update({"figure.constrained_layout.use": True})
mpl.rcParams['axes.unicode_minus'] = False

In [3]:
from google.colab import drive

drive.mount('/content/drive/', force_remount=True)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
#cd /content/drive/MyDrive/AI3_prjct2_aladin/

In [4]:
cd /content/drive/MyDrive/WASSUP-ESTsoft-AI/project/project2/

[Errno 2] No such file or directory: '/content/drive/MyDrive/WASSUP-ESTsoft-AI/project/project2/'
/home/doeun/code/AI/ESTSOFT2024/workspace/2.project_text/transformer_models/research


In [5]:
# 로컬에서

plt.rc("font", family = "D2Coding")
plt.rcParams["axes.unicode_minus"] = False

In [6]:
PRJCT_PATH = '/home/doeun/code/AI/ESTSOFT2024/workspace/2.project_text/transformer_models/'
#PRJCT_PATH = '/content/drive/MyDrive/WASSUP-ESTsoft-AI/project/project2/'
#PRJCT_PATH = '/content/drive/MyDrive/AI3_prjct2_aladin/aladin_usedbook/'
save_dir = 'processed/model_input'
dir_path = os.path.join(PRJCT_PATH,save_dir)
#dir_path = './'

In [7]:
dir_path

'/home/doeun/code/AI/ESTSOFT2024/workspace/2.project_text/transformer_models/processed/model_input'

In [8]:
import sys
sys.path.append(PRJCT_PATH)

In [9]:
RSLT_DIR = PRJCT_PATH + 'processed/'

bookinfo_name = 'bookinfo_ver{}_single.csv'.format(2.0)
bookinfo_path = os.path.join(RSLT_DIR,bookinfo_name)

sys.path.append(PRJCT_PATH)
from module_aladin.file_io import load_pkl, save_pkl
from module_aladin.data_process import pd_datetime_2_datenum

import itertools

In [10]:
def set_corpus_size(freq,size_feat,mode):
    # 입력받은 mode와 size_feat에 따라 size 크기 결정
    if mode == 'uniform':
        cond = freq['counts']>=freq['counts'].iloc[size_feat]
        size = np.sum(cond)
    elif mode =='ths':
        cond = freq[freq['counts'] > size_feat]
        size = np.sum(cond)
    else :
        if size_feat == None : size = len(freq)
#        elif size_feat > len(data) : size = len(freq)
        else : size = size_feat
    return size

def make_encoding_by_freq(freq,null_val='[PAD]',size_feat=None,mode=None):
    #빈도수 기반 정수 인코딩 dict 만들기
    # freq : token 별 등장 빈도 (value_count), size_feat : size관련 변수(max_size, ths등), mode : size 결정 방법
    df_freq = pd.DataFrame(freq).T
    df_freq = df_freq.rename(columns={0:'token',1:'counts'})
    temp = df_freq.sort_values(by='counts',ascending=False)
    size = set_corpus_size(temp,size_feat,mode)
    temp = temp.iloc[:size]
    temp['val'] = np.arange(size)+1
    temp2 = temp.set_index('token').to_dict()
    map_token_encode = temp2['val']
    map_token_encode[null_val]=0
    return map_token_encode

def encode_tokens(map_token,x,oov=True):
    oov_val = len(map_token)+1 if oov else 0 
    return map_token[x] if x in map_token else oov_val

def make_author_encode_map(bookinfo,ths_author):
    pvtb = pd.pivot_table(data=bookinfo,index='Author',values='SalesPoint',aggfunc=np.sum)
    pvtb = pvtb.sort_values(by='SalesPoint',ascending=False)
    author_top_k= pvtb[pvtb['SalesPoint']>=ths_author].index
    encode_author = pd.DataFrame({'author' : author_top_k.values,'val':np.arange(1,len(author_top_k)+1)})
    encode_author = encode_author.set_index('author')
    return encode_author.to_dict()['val']

def make_publshr_encode_map(publshr_data,ths_publshr):
    stats = publshr_data.value_counts().sort_values(ascending=False)
    top_k_val = stats.iloc[ths_publshr]
    publshr_top_k = list(stats[stats >= top_k_val].index)
    return {
        publshr : n+1
        for n,publshr in enumerate(publshr_top_k)
    }
    

def make_store_encode_map(store_data):
    stores= store_data.value_counts().sort_values(ascending=False)
    return {
        place : n+1
        for n,place in enumerate(stores.index)
    }

In [11]:
#file_name = 'bookinfo_ver{}.csv'.format(1.0)
#file_path = os.path.join(RSLT_DIR,file_name)
bookinfo_raw = pd.read_csv(bookinfo_path)

In [12]:
cols = bookinfo_raw.columns.to_list()
x_idxs, y_idx = [0,2,3,4,5,6,9,10], 7
x_cols = [cols[i] for i in x_idxs]
y_col = cols[y_idx]

data_X, data_y  = bookinfo_raw[x_cols], bookinfo_raw[y_col]

In [37]:
from sklearn.model_selection import train_test_split

X_data, X_tst, y_data, y_tst = train_test_split(data_X,data_y,test_size=0.2,random_state=329)
X_trn, X_vld, y_trn, y_vld = train_test_split(X_data,y_data,test_size=0.2,random_state=329)

display(X_trn.columns)
display(X_trn.shape, y_trn.shape)
display(X_vld.shape, y_vld.shape)
display(X_tst.shape, y_tst.shape)

Index(['BName', 'BName_sub', 'Author', 'Publshr', 'Author_mul', 'Pdate',
       'SalesPoint', 'Category'],
      dtype='object')

(98835, 8)

(98835,)

(24709, 8)

(24709,)

(30887, 8)

(30887,)

In [38]:
data_dict = {
    'trn': {
        'X': X_trn,
        'y': y_trn
        },
    'vld':{
        'X': X_vld,
        'y': y_vld
        },
    'tst':{
        'X': X_tst,
        'y': y_tst
        
    }
}

In [39]:
display(data_dict['trn']['X'].info())
display(data_dict['vld']['X'].info())
display(data_dict['tst']['X'].info())

<class 'pandas.core.frame.DataFrame'>
Index: 98835 entries, 84226 to 37109
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   BName       98834 non-null  object
 1   BName_sub   5416 non-null   object
 2   Author      98835 non-null  object
 3   Publshr     98835 non-null  object
 4   Author_mul  98835 non-null  bool  
 5   Pdate       98835 non-null  int64 
 6   SalesPoint  98835 non-null  int64 
 7   Category    98835 non-null  object
dtypes: bool(1), int64(2), object(5)
memory usage: 6.1+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 24709 entries, 134680 to 143289
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   BName       24709 non-null  object
 1   BName_sub   1299 non-null   object
 2   Author      24709 non-null  object
 3   Publshr     24709 non-null  object
 4   Author_mul  24709 non-null  bool  
 5   Pdate       24709 non-null  int64 
 6   SalesPoint  24709 non-null  int64 
 7   Category    24709 non-null  object
dtypes: bool(1), int64(2), object(5)
memory usage: 1.5+ MB


None

<class 'pandas.core.frame.DataFrame'>
Index: 30887 entries, 30824 to 33317
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   BName       30887 non-null  object
 1   BName_sub   1573 non-null   object
 2   Author      30887 non-null  object
 3   Publshr     30887 non-null  object
 4   Author_mul  30887 non-null  bool  
 5   Pdate       30887 non-null  int64 
 6   SalesPoint  30887 non-null  int64 
 7   Category    30887 non-null  object
dtypes: bool(1), int64(2), object(5)
memory usage: 1.9+ MB


None

In [40]:
#apply tokenizer
cols_freq = ['Author','Publshr']

In [41]:
#아래 ths 는 EDA 결과 제가 자의적으로 정한 내용
bookinfo = data_dict['trn']['X']
ths_author = int(np.round(len(bookinfo)/500)*75)
ths_publshr = int(np.round(len(bookinfo)/500)*5)

map_author_encode = make_author_encode_map(bookinfo[['Author','SalesPoint']],ths_author)
map_publshr_encode = make_publshr_encode_map(bookinfo['Publshr'],ths_publshr)

encode_maps = {
    'Author' : lambda x : encode_tokens(map_author_encode,x,oov=False),
    'Publshr' : lambda x : encode_tokens(map_publshr_encode,x,oov=False),
}

In [42]:
for mode,data in data_dict.items():
    for col in tqdm(cols_freq):
        data['X'][col] = data['X'][col].map(encode_maps[col]).astype(str)

100%|██████████| 2/2 [00:00<00:00, 24.03it/s]


In [43]:
X_col_dict = {
 'BName':'제목',
 'BName_sub':'부제',
 'Category':'분류',
 'Pdate':'출간일',
 'Author':'저자 인지도 순위',
 'Author_mul':'공동 저자 유무',
 'Publshr':'출판사 인지도 순위',
}
#kor_exist={True:'있음',False:'없음'}
kor_exist={True:'참',False:'거짓'}

In [44]:
ranks = ['Author','Publshr']

#oov_val = '.'
oov_val = '순위 외'

for mode,sample in tqdm(data_dict.items()):
    for col in ranks:
        cond = data_dict[mode]['X'][col] == '0'
        data_dict[mode]['X'].loc[cond,col] = oov_val

100%|██████████| 3/3 [00:00<00:00, 102.46it/s]


In [45]:
bool_key = True 

#encode X
X_encoded=dict()
for mode,sample in tqdm(data_dict.items()):
    X_mode = sample['X'].copy()
    #padding and encoding
    encoded = pd.DataFrame(index=X_mode.index)
    X_mode['BName_sub'] = X_mode['BName_sub'].fillna('.')
    X_mode['Pdate'] = X_mode['Pdate'].astype(str).apply(lambda x :f'{x[:4]}년 {x[4:6]}월 {x[6:]}일')
    X_mode['Author_mul'] = X_mode['Author_mul'].map(kor_exist)
    for col,sub in X_col_dict.items():
        if bool_key : encoded[col] = X_mode[col].apply(lambda x: f'{sub} : {x}') 
        else : encoded[col] = X_mode[col].astype(str)
    temp = '[CLS]' + encoded[X_col_dict.keys()].agg(' [SEP] '.join, axis=1) + ' [SEP]'
    X_encoded[mode] = temp.values

100%|██████████| 3/3 [00:02<00:00,  1.11it/s]


In [46]:
data_dict['trn']['X'].iloc[74310]

BName         프러포즈는 필요없어
BName_sub            NaN
Author              순위 외
Publshr             순위 외
Author_mul          True
Pdate           20070927
SalesPoint            68
Category         소설/시/희곡
Name: 46321, dtype: object

In [47]:
X_encoded['trn'][74310]

'[CLS]제목 : 프러포즈는 필요없어 [SEP] 부제 : . [SEP] 분류 : 소설/시/희곡 [SEP] 출간일 : 2007년 09월 27일 [SEP] 저자 인지도 순위 : 순위 외 [SEP] 공동 저자 유무 : 참 [SEP] 출판사 인지도 순위 : 순위 외 [SEP]'

In [48]:
X_encoded['vld'][4310]

'[CLS]제목 : 2022 최신판 MBC 기본직무소양평가 최종모의고사 6회분 + 무료NCS특강 [SEP] 부제 : . [SEP] 분류 : 수험서/자격증 [SEP] 출간일 : 2022년 08월 25일 [SEP] 저자 인지도 순위 : 1556 [SEP] 공동 저자 유무 : 거짓 [SEP] 출판사 인지도 순위 : 21 [SEP]'

In [49]:
data_dict['trn']['X'].iloc[74310,3], '268' in data_dict['trn']['X']['Publshr'].values

('순위 외', True)

In [52]:
for mode,data in X_encoded.items():
    temp = np.array(list(map(len,data)))
    print(mode,temp.max(),temp.min())

trn 250 132
vld 257 134
tst 236 134


In [53]:
courpus_size_in = 31150
max_len = 64+len(X_col_dict)*3+1

max_len

86

In [54]:
X_coded = {
    mode : data
    for mode, data in X_encoded.items()
}
X_coded['info'] ={
    'corpus_size' : courpus_size_in,
    'max_len' : 100
}

### encode y

In [55]:
def remain_k_digits(n,k):
  r = np.floor(np.log10(n))
  temp = np.round(n/(10**r),k-1)
  temp = np.round(temp*(10**r),0)
  return (temp).astype(int)

def make_val_keys(data):
  lbnd = data.min()//1000
  ubnd = data.max()//1000
  temp = np.unique(remain_k_digits(np.arange(lbnd,ubnd+2),2)*1000)
  return temp[1:]

def encode_tokens(map_token,x,oov=True):
    oov_val = len(map_token)+1 if oov else 0
    return map_token[x] if x in map_token else oov_val

def encode_to_2digit(data,keys):
  val = remain_k_digits(data,2)
  val[val < keys.min()] = keys.min()
  val[val > keys.max()] = keys.max()

  cond = val < 10000
  val[cond] = remain_k_digits(val[cond],1)
  encode_map= {
      k : i for i,k in enumerate(keys)
  }
  encode_1line = lambda y : list(map(lambda x : encode_tokens(encode_map,x),y))
  return np.apply_along_axis(encode_1line,0,val), encode_map


In [56]:
data = data_dict['trn']['y']
val_keys = make_val_keys(data)
encoded,encode_map = encode_to_2digit(data,val_keys)

/tmp/ipykernel_1989/3157627052.py:2: RuntimeWarning: divide by zero encountered in log10
  r = np.floor(np.log10(n))
/tmp/ipykernel_1989/3157627052.py:3: RuntimeWarning: invalid value encountered in divide
  temp = np.round(n/(10**r),k-1)
/tmp/ipykernel_1989/3157627052.py:5: RuntimeWarning: invalid value encountered in cast
  return (temp).astype(int)


In [57]:
encoded.min()

0

In [58]:
decode_map = {v:k for k,v in encode_map.items()}

In [59]:
def fill_cls_freq(cls_freq,corpus_size):
    keys = list(filter(lambda x : x not in cls_freq[0],range(corpus_size)))
    temp = [keys,np.zeros(len(keys))]
    updated = np.hstack([cls_freq,np.array(temp)]).astype(np.int32)
    return updated[:,updated[0].argsort()]

In [60]:
from collections import defaultdict
y_coded = dict() 
for mode,sample in data_dict.items():
    coded,_ = encode_to_2digit(sample['y'],val_keys)
    y_coded[mode]= {
        'value': sample['y'].to_numpy(),
        'coded': coded, }

corpus_size_out=len(encode_map)
y_trn_freq = np.unique(y_coded['trn']['coded'],return_counts=True)
y_coded['info']= {
    'encode':{'decode_map':  {'map':decode_map,
                    'freq':fill_cls_freq(y_trn_freq,corpus_size_out)},
    },
    'corpus_size':  corpus_size_out,
}

    

In [61]:
for mode,sample in y_coded.items():
    if mode == 'info' : continue
    temp = sample['coded']
    print(mode,temp.min(),temp.max())

trn 0 114
vld 0 114
tst 0 112


### encode log y

In [50]:
# encode y

def encode_log(x):
    str_x = f'{np.log10(x):.4f}'.split('.')
    temp = [str_x[0]]+list(str_x[1])
    return ['[SOS]']+temp+['[EOS]']

tkn_lists = data_dict['trn']['y'].apply(encode_log).values
#y_tkn_vals = np.array(list(itertools.chain(*tkn_lists)))
y_tkn_vals = np.concatenate(tkn_lists,axis=0)
y_tkn_freq = np.unique(y_tkn_vals,return_counts=True)
map_y_tkns = make_encoding_by_freq(y_tkn_freq)
encode_y_1line = lambda x : list(map(lambda y : encode_tokens(map_y_tkns,y,oov=False),x))

display(y_tkn_freq)
display(map_y_tkns)

(array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '[EOS]', '[SOS]'],
       dtype='<U5'),
 array([ 55809,  66163,  46786,  69609, 109232,  28181,  23211,  43861,
         16278,  46735, 101173, 101173]))

{'4': 1,
 '[EOS]': 2,
 '[SOS]': 3,
 '3': 4,
 '1': 5,
 '0': 6,
 '2': 7,
 '9': 8,
 '7': 9,
 '5': 10,
 '6': 11,
 '8': 12,
 '[PAD]': 0}

In [52]:
decode_map = {v:k for k,v in map_y_tkns.items()}
sen_tkns = ['[SOS]','[EOS]','[PAD]',]
sentkns_dict = {
    tkn : map_y_tkns[tkn]
    for tkn in sen_tkns
}

In [53]:
def fill_cls_freq(cls_freq,corpus_size):
    keys = list(filter(lambda x : x not in cls_freq[0],range(corpus_size)))
    temp = [keys,np.zeros(len(keys))]
    updated = np.hstack([cls_freq,np.array(temp)]).astype(np.int32)
    return updated[:,updated[0].argsort()]

In [55]:
from collections import defaultdict
y_coded = dict() 
for mode,sample in data_dict.items():
    temp = sample['y'].apply(encode_log)
    padded = pad_sequences(temp,padding='post',
                                   maxlen=y_max_len,
                                   value='[PAD]',dtype=object)
    intrmd = np.apply_along_axis(encode_y_1line,0,padded)
    y_coded[mode]= {
        'value': sample['y'].to_numpy(),
        'coded': intrmd.astype(np.int32), }

corpus_size_out=13
y_trn_freq = np.unique(y_coded['trn']['coded'],return_counts=True)
y_coded['info']= {
    'decode_map':  {'map':decode_map,
                    'freq':fill_cls_freq(y_trn_freq,corpus_size_out)},
    'corpus_size':  corpus_size_out,
    'tkn': sentkns_dict ,
    'max_len': y_max_len,
}

    

In [42]:
decode_map

{0: 1000,
 1: 2000,
 2: 3000,
 3: 4000,
 4: 5000,
 5: 6000,
 6: 7000,
 7: 8000,
 8: 9000,
 9: 10000,
 10: 11000,
 11: 12000,
 12: 13000,
 13: 14000,
 14: 15000,
 15: 16000,
 16: 17000,
 17: 18000,
 18: 19000,
 19: 20000,
 20: 21000,
 21: 22000,
 22: 23000,
 23: 24000,
 24: 25000,
 25: 26000,
 26: 27000,
 27: 28000,
 28: 29000,
 29: 30000,
 30: 31000,
 31: 32000,
 32: 33000,
 33: 34000,
 34: 35000,
 35: 36000,
 36: 37000,
 37: 38000,
 38: 39000,
 39: 40000,
 40: 41000,
 41: 42000,
 42: 43000,
 43: 44000,
 44: 45000,
 45: 46000,
 46: 47000,
 47: 48000,
 48: 49000,
 49: 50000,
 50: 51000,
 51: 52000,
 52: 53000,
 53: 54000,
 54: 55000,
 55: 56000,
 56: 57000,
 57: 58000,
 58: 59000,
 59: 60000,
 60: 61000,
 61: 62000,
 62: 63000,
 63: 64000,
 64: 65000,
 65: 66000,
 66: 67000,
 67: 68000,
 68: 69000,
 69: 70000,
 70: 71000,
 71: 72000,
 72: 73000,
 73: 74000,
 74: 75000,
 75: 76000,
 76: 77000,
 77: 78000,
 78: 79000,
 79: 80000,
 80: 81000,
 81: 82000,
 82: 83000,
 83: 84000,
 84: 85000,

In [62]:
RSLT_DIR

'/home/doeun/code/AI/ESTSOFT2024/workspace/2.project_text/transformer_models/processed/'

In [64]:
ls {RSLT_DIR}/model_input

537.49s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


BERT_cls-single.v1.0_st-0_X_info.pkl   encodedXy.v2.5_st-0_X_tst.pkl
BERT_cls-single.v1.0_st-0_X_trn.pkl    encodedXy.v2.5_st-0_X_vld.pkl
BERT_cls-single.v1.0_st-0_X_tst.pkl    encodedXy.v2.5_st-0_y_info.pkl
BERT_cls-single.v1.0_st-0_X_vld.pkl    encodedXy.v2.5_st-0_y_trn.pkl
BERT_cls-single.v1.0_st-0_y_info.pkl   encodedXy.v2.5_st-0_y_tst.pkl
BERT_cls-single.v1.0_st-0_y_trn.pkl    encodedXy.v2.5_st-0_y_vld.pkl
BERT_cls-single.v1.0_st-0_y_tst.pkl    encodedXy.v3.0_st-0_X_info.pkl
BERT_cls-single.v1.0_st-0_y_vld.pkl    encodedXy.v3.0_st-0_X_trn.pkl
BERT_cls-single.v1.a_st-0_X_info.pkl   encodedXy.v3.0_st-0_X_tst.pkl
BERT_cls-single.v1.a_st-0_X_trn.pkl    encodedXy.v3.0_st-0_X_vld.pkl
BERT_cls-single.v1.a_st-0_X_tst.pkl    encodedXy.v3.0_st-0_y_info.pkl
BERT_cls-single.v1.a_st-0_X_vld.pkl    encodedXy.v3.0_st-0_y_trn.pkl
BERT_cls-single.v1.a_st-0_y_info.pkl   encodedXy.v3.0_st-0_y_tst.pkl
BERT_cls-single.v1.a_st-0_y_trn.pkl    encodedXy.v3.0_st-0_y_vld.pkl
BERT_cls-single.v1.a_st-0_y_tst

In [65]:
data_type = 'BERT_cls-single'
strat=0
ver='1.5'
dir_path = os.path.join(RSLT_DIR,'model_input')
for mode, x_scaled in X_coded.items():
    save_pkl(dir_path,'{}.v{}_st-{}_X_{}.pkl'.format(data_type,ver,strat,mode),x_scaled)
for mode,data in y_coded.items(): 
    save_pkl(dir_path,'{}.v{}_st-{}_y_{}.pkl'.format(data_type,ver,strat,mode),data)

In [66]:
cand = list(filter(lambda x : 'info' in x ,os.listdir(dir_path)))
cand = sorted(cand)

temp = list(map(lambda x: load_pkl(os.path.join(dir_path,x)),cand))
[(n,x.keys()) for n,x in zip(cand,temp)]

[('BERT_cls-single.v1.0_st-0_X_info.pkl',
  dict_keys(['corpus_size', 'max_len'])),
 ('BERT_cls-single.v1.0_st-0_y_info.pkl',
  dict_keys(['encode', 'corpus_size'])),
 ('BERT_cls-single.v1.5_st-0_X_info.pkl',
  dict_keys(['corpus_size', 'max_len'])),
 ('BERT_cls-single.v1.5_st-0_y_info.pkl',
  dict_keys(['encode', 'corpus_size'])),
 ('BERT_cls-single.v1.a_st-0_X_info.pkl',
  dict_keys(['corpus_size', 'max_len'])),
 ('BERT_cls-single.v1.a_st-0_y_info.pkl',
  dict_keys(['encode', 'corpus_size'])),
 ('encodedXy-single.v5.0_st-0_X_info.pkl',
  dict_keys(['corpus_size', 'max_len', 'encode'])),
 ('encodedXy-single.v5.0_st-0_y_info.pkl',
  dict_keys(['encode', 'corpus_size', 'tkn', 'max_len'])),
 ('encodedXy.v2.0_st-0_y_info.pkl', dict_keys(['decode_map', 'freq'])),
 ('encodedXy.v2.5_st-0_X_info.pkl', dict_keys(['corpus_size'])),
 ('encodedXy.v2.5_st-0_y_info.pkl',
  dict_keys(['decode_map', 'corpus_size', 'freq', 'tkn', 'max_len'])),
 ('encodedXy.v3.0_st-0_X_info.pkl', dict_keys(['corpus_siz

#### numpy,torch examples

In [164]:
np.triu(np.ones((1,7,7)),k=1)==0

array([[[ True, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True]]])

In [58]:
import torch
torch.zeros(3,3,3)

tensor([[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]])

In [166]:
a = np.triu(np.ones((2,7,3)),k=-1)


a = torch.LongTensor(a)
a

tensor([[[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]])

In [167]:
a[:,-1,:] = 2

a

tensor([[[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [0, 0, 0],
         [0, 0, 0],
         [2, 2, 2]],

        [[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [0, 0, 0],
         [0, 0, 0],
         [2, 2, 2]]])

In [175]:
a[:,-2,:] = torch.LongTensor([[1,3,2],[9,8,7]])
a

tensor([[[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [5, 7, 8],
         [1, 3, 2],
         [2, 2, 2]],

        [[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [5, 7, 8],
         [9, 8, 7],
         [2, 2, 2]]])

In [177]:
a[:,-3] = torch.LongTensor([5,7,8])
a

tensor([[[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [5, 7, 8],
         [1, 3, 2],
         [2, 2, 2]],

        [[1, 1, 1],
         [1, 1, 1],
         [0, 1, 1],
         [0, 0, 1],
         [5, 7, 8],
         [9, 8, 7],
         [2, 2, 2]]])

In [178]:
a.max(dim=-1)

torch.return_types.max(
values=tensor([[1, 1, 1, 1, 8, 3, 2],
        [1, 1, 1, 1, 8, 9, 2]]),
indices=tensor([[0, 0, 1, 2, 2, 1, 0],
        [0, 0, 1, 2, 2, 0, 0]]))

In [180]:
a[:,:,-2] = torch.LongTensor([2,5,6,3,1,9,6])
a

tensor([[[1, 2, 1],
         [1, 5, 1],
         [0, 6, 1],
         [0, 3, 1],
         [5, 1, 8],
         [1, 9, 2],
         [2, 6, 2]],

        [[1, 2, 1],
         [1, 5, 1],
         [0, 6, 1],
         [0, 3, 1],
         [5, 1, 8],
         [9, 9, 7],
         [2, 6, 2]]])

In [183]:
a

tensor([[[ 1,  2,  1],
         [ 1,  5,  1],
         [ 0,  6,  1],
         [ 0,  3,  1],
         [ 5,  1,  8],
         [ 1,  9,  2],
         [ 2,  6, 10]],

        [[ 1,  2,  1],
         [ 1,  5,  1],
         [ 0,  6,  1],
         [ 0,  3,  1],
         [ 5,  1,  8],
         [ 9,  9,  7],
         [ 2,  6, 10]]])

In [186]:
a[-1,-1,-1] = 7
a

tensor([[[ 1,  2,  1],
         [ 1,  5,  1],
         [ 0,  6,  1],
         [ 0,  3,  1],
         [ 5,  1,  8],
         [ 1,  9,  2],
         [ 2,  6, 10]],

        [[ 1,  2,  1],
         [ 1,  5,  1],
         [ 0,  6,  1],
         [ 0,  3,  1],
         [ 5,  1,  8],
         [ 9,  9,  7],
         [ 2,  6,  7]]])

In [187]:
a[:,-1,:].max(dim=-1)

torch.return_types.max(
values=tensor([10,  7]),
indices=tensor([2, 2]))